In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize
from termcolor import colored

In [26]:
df_reactions = pd.read_parquet("./data/reactions.parquet")
df_metrics = pd.read_parquet("./reactions_sample_5000_with_metrics.parquet")
df = df_metrics.merge(df_reactions, on="id", how="left")
del df_reactions, df_metrics
df.head()

94294


,id,novelty_mattr,local_coherence,divergent_thinking,timestamp,model_a_name,model_b_name,refers_to_model,msg_index,opening_msg,...,creative,complete,clear_formatting,incorrect,superficial,instructions_not_followed,model_pair_name,msg_rank,question_id,system_prompt
0,407883,0.616022,0.584729,0.645262,2026-01-12 11:08:39.979662,gemini-2.5-flash,mistral-small-2506,mistral-small-2506,1,Je voudrais une infographie concernant la léga...,...,False,False,False,False,True,False,"[gemini-2.5-flash, mistral-small-2506]",0,34d99aef40a541a9a519aa2138b5337b-87c3ed9179844...,NaN
1,196058,1.000000,1.000000,0.000000,2025-04-13 20:07:45.202795,gemma-2-9b-it,hermes-3-llama-3.1-405b,gemma-2-9b-it,1,Rédige-moi 10 questions en espagnol de compréh...,...,False,False,False,True,False,False,"[gemma-2-9b-it, hermes-3-llama-3.1-405b]",0,a2aa62a6fc1846feb5ca00bc97106964-5fab3f723ae64...,NaN
2,166222,0.619663,0.536445,0.626564,2025-03-18 07:49:56.046668,llama-3.3-70b,llama-3.1-405b,llama-3.3-70b,11,"Bonjour, quels sources d'informations fiables ...",...,False,False,False,False,False,False,"[llama-3.1-405b, llama-3.3-70b]",5,386056c19a4c4282a14badb2e9b2cfad-309833adb6f14...,
3,43777,0.514634,0.770187,0.526662,2025-02-09 11:01:51.273174,qwen2.5-coder-32b-instruct,hermes-3-llama-3.1-405b,qwen2.5-coder-32b-instruct,25,Peux-tu me transformer l'ensemble du texte sui...,...,False,False,True,False,False,False,"[hermes-3-llama-3.1-405b, qwen2.5-coder-32b-in...",12,775f1b81fedc47e0a49152492bcce88f-a44d9e0e83a94...,
4,182849,0.749483,0.428732,0.554014,2025-03-31 06:43:52.920557,llama-3.1-8b,deepseek-v3-chat,deepseek-v3-chat,1,comment la photosynthèse permet-elle de faire\...,...,False,False,True,False,False,False,"[deepseek-v3-chat, llama-3.1-8b]",0,f077840acb3743e78d8ea1aae80d64cf-98dd163342bc4...,NaN


## Validation de convergence

In [49]:
metric_cols = ["novelty_mattr", "local_coherence", "divergent_thinking"]

# Compute correlation between "creative" and each metric
for metric in metric_cols:
    metric_values = list(df[metric].astype(int))
    creative_values = list(df["creative"])

    # Compute basic correlation
    corr, _ = pearsonr(metric_values, creative_values)
    print(f"Correlation between 'creative' and '{metric}': {corr:.4f} (pearson)")

    if abs(corr) > 0.4:
        print(colored(f"  -> Strong correlation detected for '{metric}'", "green"))
    else:
        print(colored(f"  -> No strong correlation detected for '{metric}'", "red"))

Correlation between 'creative' and 'novelty_mattr': 0.0017 (pearson)
  -> No strong correlation detected for 'novelty_mattr'
Correlation between 'creative' and 'local_coherence': -0.0311 (pearson)
  -> No strong correlation detected for 'local_coherence'
Correlation between 'creative' and 'divergent_thinking': -0.0038 (pearson)
  -> No strong correlation detected for 'divergent_thinking'


### Analyse des l'indépendance espérée entre certaines variables et la variable "creative"

In [ ]:
import os
import re
from tqdm import tqdm

df["response_length"] = df["response_content"].str.len()

np.int64(1)

In [48]:
independant_vars = ["clear_formatting", "incorrect", "response_length"]

for variable in independant_vars:
    variable_values = list(df[variable].astype(int))
    creative_values = list(df["creative"])

    # Compute basic correlation
    corr, _ = pearsonr(variable_values, creative_values)
    print(f"Correlation between 'creative' and '{variable}': {corr:.4f} (pearson)")
    if abs(corr) < 0.2:
        print(colored(f"\t=> Variable '{variable}' is relatively independent from 'creative' (|corr| < 0.2)", "green"))
    else:
        print(colored(f"\t=> Variable '{variable}' is not independent from 'creative' (|corr| >= 0.2)", "red"))

Correlation between 'creative' and 'clear_formatting': 0.1106 (pearson)
	=> Variable 'clear_formatting' is relatively independent from 'creative' (|corr| < 0.2)
Correlation between 'creative' and 'incorrect': -0.0924 (pearson)
	=> Variable 'incorrect' is relatively independent from 'creative' (|corr| < 0.2)
Correlation between 'creative' and 'response_length': 0.0771 (pearson)
	=> Variable 'response_length' is relatively independent from 'creative' (|corr| < 0.2)
